[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/dif/u2_limite.ipynb)

# Dif. U2 · Concepto de límite
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 2 del bloque de cálculo diferencial. Tiene una sección por subtema, en el mismo orden que el libro:

1. Aproximación numérica por tablas desde ambos lados
2. Lectura de límites en gráficas y límites laterales
3. Límites en el infinito
4. Álgebra de límites y la forma indeterminada $0/0$
5. Teorema del sándwich y límites trigonométricos
6. Continuidad, tipos de discontinuidad y método de bisección

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, calcula a mano el caso que se indica y escribe tu resultado. Una tabla o una gráfica *sugieren* un límite; el álgebra o un teorema lo confirman.

**Etiqueta del repositorio.** `DIF-U2` (bloque, unidad). Las celdas de código de cada sección vienen de `dif/u2_limite/NN_*.py`; los fragmentos que el libro imprime, de `dif/u2_limite/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir expresiones usa `x` como variable, `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)` para raíces, `abs(...)` para el valor absoluto, `exp(...)` para $e^x$ y `log(...)` para el logaritmo natural. Los ángulos de `sin`, `cos` y `tan` van en **radianes**.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8"

import sys, math, re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor)
from sympy.calculus.util import continuous_domain

print("Python", sys.version.split()[0], "| numpy", np.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)

_LOCAL = {"x": x, "sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log,
          "pi": sp.pi, "sin": sp.sin, "cos": sp.cos, "tan": sp.tan, "E": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor)


def parsear(texto):
    """Convierte un texto en expresión de sympy en la variable x, o explica qué salió mal."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una expresión, por ejemplo: (x**3 - 1)/(x - 1)")
    desconocidos = sorted(set(re.findall(r"[A-Za-z_]\w*", texto)) - set(_LOCAL))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa x como variable y, si hace falta, "
                         "sqrt, abs, exp, log (natural), sin, cos, tan o pi.")
    try:
        expr = parse_expr(texto, local_dict=_LOCAL, global_dict=dict(_GLOBAL),
                          transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la expresión. Usa x como variable, * para multiplicar, "
                         "** o ^ para potencias y sqrt(...) para raíces.") from err
    if expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Hay una función que no conozco. Las permitidas son sqrt, abs, exp, log, sin, cos y tan.")
    libres = {str(s) for s in expr.free_symbols} - {"x"}
    if libres:
        raise ValueError(f"Símbolos no permitidos: {sorted(libres)}. Solo puedes usar x.")
    return expr


def numero(texto):
    """Lee un número escrito como 2, -0.5, 1/3, pi/2 o sqrt(2)."""
    try:
        v = sp.sympify(str(texto).replace("^", "**"), locals={"pi": sp.pi, "sqrt": sp.sqrt, "E": sp.E})
    except (sp.SympifyError, TypeError):
        raise ValueError(f"'{texto}' no es un número.") from None
    if v.free_symbols or not v.is_real:
        raise ValueError(f"'{texto}' no es un número real.")
    return v


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas y devuelve texto."""
    if valor is None:
        return "no existe"
    if valor in (sp.oo, sp.S.Infinity) or (isinstance(valor, float) and valor == math.inf):
        return "+∞"
    if valor in (-sp.oo, sp.S.NegativeInfinity) or (isinstance(valor, float) and valor == -math.inf):
        return "-∞"
    v = float(valor)
    if math.isnan(v):
        return "no definida"
    if v == 0:
        return "0"
    return np.format_float_positional(v, precision=n, unique=False, fractional=False, trim="-")


def a_texto(valor):
    """Límite de sympy en texto: número exacto si es racional, ±∞ o 'no existe'."""
    if valor is None or valor is sp.nan:
        return "no existe"
    if valor == sp.oo:
        return "+∞"
    if valor == -sp.oo:
        return "-∞"
    if valor == sp.zoo:
        return "∞ sin signo (no existe)"
    if isinstance(valor, sp.AccumBounds):
        return "no existe (oscila)"
    return str(valor) if valor.is_Rational else f"{valor} ≈ {cifras(valor, 6)}"


def conjunto_a_texto(s):
    """Intervalos de sympy en notación de intervalos."""
    if s == sp.S.Reals:
        return "(-∞, ∞) = ℝ"
    if s == sp.S.EmptySet:
        return "∅ (vacío)"
    if isinstance(s, sp.Union):
        return " ∪ ".join(conjunto_a_texto(t) for t in sorted(s.args, key=lambda t: float(t.inf)))
    if isinstance(s, sp.Interval):
        f = lambda p: ("∞" if p == sp.oo else "-∞" if p == -sp.oo
                       else str(p) if p.is_Rational else "≈" + cifras(float(p), 4))
        return ("(" if s.left_open else "[") + f(s.inf) + ", " + f(s.sup) + (")" if s.right_open else "]")
    return str(s)


def evaluar(f, v):
    """f(v) en punto flotante; nan si f no está definida en v."""
    with np.errstate(all="ignore"):
        try:
            r = complex(f(v))
        except (ZeroDivisionError, ValueError, OverflowError):
            return math.nan
    return r.real if abs(r.imag) < 1e-12 else math.nan


def cerca(u, v, tol=1e-9):
    return abs(u - v) < tol


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False

## 1. Aproximación numérica por tablas desde ambos lados

Para explorar $\lim_{x\to a} f(x)$ se evalúa $f$ en $a-h$ y en $a+h$ con $h=0.1,\ 0.01,\ 0.001,\dots$ El valor $f(a)$ no interviene: muchas veces ni siquiera existe.

La calculadora siguiente arma esa tabla. Si los dos lados se acercan al mismo número, la tabla **sugiere** ese límite. Con $h$ menor que $10^{-5}$ aparece otro riesgo, el redondeo de la máquina (Laboratorio del Error 2.2): la calculadora te avisa y no usa esos renglones para su lectura.

In [ ]:
def tabla_lados(texto, a, k=4):
    """Filas (h, f(a-h), f(a+h)) con h = 10^-1 ... 10^-k. Valida las entradas."""
    f_expr = parsear(texto)
    a = float(numero(a))
    if not 1 <= int(k) <= 12:
        raise ValueError("Elige entre 1 y 12 renglones.")
    f = sp.lambdify(x, f_expr, "numpy")
    filas = []
    for j in range(1, int(k) + 1):
        h = 10.0**-j
        filas.append((h, evaluar(f, a - h), evaluar(f, a + h)))
    return filas


def lectura(filas, tol=1e-3):
    """Qué sugiere la tabla, usando solo los renglones con h >= 1e-5."""
    utiles = [r for r in filas if r[0] >= 1e-5 * 0.999] or filas[:1]
    _, izq, der = utiles[-1]
    if math.isnan(izq) or math.isnan(der):
        return "f no está definida de algún lado: revisa el dominio."
    if abs(izq) > 1e6 or abs(der) > 1e6:
        return "los valores crecen sin límite: posible límite infinito (subtema 2.2)."
    if abs(izq - der) < tol * max(1, abs(izq)):
        return f"los dos lados se acercan a ≈ {cifras((izq + der) / 2, 4)} (sugerencia, no demostración)."
    return f"los lados no coinciden (≈ {cifras(izq, 4)} y ≈ {cifras(der, 4)}): el límite no existe."


def calculadora_lados(texto, a, k, n_cifras):
    try:
        filas = tabla_lados(texto, a, k)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"f(x) = {texto}   cerca de a = {a}   (valores redondeados a {n_cifras} cifras significativas)")
    print(f"{'h':>8} | {'f(a - h)':>16} | {'f(a + h)':>16}")
    for h, izq, der in filas:
        print(f"{h:>8.0e} | {cifras(izq, n_cifras):>16} | {cifras(der, n_cifras):>16}")
    if filas[-1][0] < 1e-5 * 0.999:
        print("Aviso: con h < 1e-5 el redondeo de la máquina puede dominar (Laboratorio del Error 2.2);")
        print("       la lectura usa solo los renglones con h >= 1e-5.")
    print("Lectura:", lectura(filas))


widgets.interact(calculadora_lados,
    texto=widgets.Text(value="(x**3 - 1)/(x - 1)", description="f(x) ="),
    a=widgets.Text(value="1", description="a ="),
    k=widgets.IntSlider(value=4, min=1, max=12, description="renglones"),
    n_cifras=widgets.IntSlider(value=7, min=1, max=10, description="cifras sig."));

**La semilla, otra vez.** La rapidez media de la pelota entre $t=2$ y $t=2+h$ es $\dfrac{s(2+h)-s(2)}{h}$ con $s(t)=4.9t^2$. En la tabla, la variable se llama $x$ (es la $h$ de la semilla) y el punto es $a=0$. Compara con la predicción que anotaste.

In [ ]:
calculadora_lados("(4.9*(2 + x)**2 - 4.9*4)/x", "0", 4, 6)

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
def ultimo_par(texto, a, k=4):
    return tabla_lados(texto, a, k)[-1][1:]

PRUEBAS_1 = [
    ("(x^3-1)/(x-1) cerca de 1: los dos lados tienden a 3",
     lambda: all(cerca(v, 3, 1e-3) for v in ultimo_par("(x**3 - 1)/(x - 1)", "1"))),
    ("f(0.9) = 2.71 y f(1.1) = 3.31",
     lambda: [round(v, 6) for v in tabla_lados("(x**3 - 1)/(x - 1)", "1", 1)[0][1:]] == [2.71, 3.31]),
    ("abs(x)/x cerca de 0: lados -1 y 1, la lectura dice que no existe",
     lambda: ultimo_par("abs(x)/x", "0") == (-1.0, 1.0) and "no existe" in lectura(tabla_lados("abs(x)/x", "0"))),
    ("semilla: la rapidez media tiende a 19.6 m/s",
     lambda: all(cerca(v, 19.6, 1e-3) for v in ultimo_par("(4.9*(2 + x)**2 - 4.9*4)/x", "0"))),
    ("factor P/A con n = 12 tiende a 12 cuando i -> 0",
     lambda: all(cerca(v, 12, 1e-2) for v in ultimo_par("(1 - (1 + x)**(-12))/x", "0"))),
    ("con h = 1e-9, (1 - cos x)/x^2 da 0 por redondeo (aviso del Laboratorio 2.2)",
     lambda: tabla_lados("(1 - cos(x))/x**2", "0", 9)[-1][1] == 0.0),
    ("una entrada con otra letra se rechaza con mensaje", lambda: _rechaza(lambda: tabla_lados("t**2", "0"))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Calcula a mano $f(0.9)$ y $f(1.1)$ para $f(x)=\dfrac{x^3-1}{x-1}$ (sin calculadora: primero $0.9^3$). Escribe tus resultados y compáralos con la tabla. Anota: ¿cuál lado queda por debajo de 3 y cuál por arriba?

In [ ]:
mi_f_09, mi_f_11 = None, None          # por ejemplo: 2.71, 3.31

esperado = tabla_lados("(x**3 - 1)/(x - 1)", "1", 1)[0][1:]
print("La calculadora da:", [round(v, 6) for v in esperado])
for nombre, mio, ref in zip(("f(0.9)", "f(1.1)"), (mi_f_09, mi_f_11), esperado):
    print(nombre, "->", "falta tu cálculo" if mio is None else ("coincide" if cerca(mio, ref, 1e-6) else "NO coincide"))

## 2. Lectura de límites en gráficas y límites laterales

Una función por partes usa una regla a la izquierda de $a$ y otra a la derecha. El límite por la izquierda, $\lim_{x\to a^-}f(x)$, solo mira la regla de la izquierda; el límite por la derecha, $\lim_{x\to a^+}f(x)$, solo la de la derecha. El límite existe cuando los dos existen y son iguales. El valor $f(a)$ es un dato aparte.

La calculadora recibe las dos reglas, el punto $a$ y el valor $f(a)$ (o la palabra `no`, si $f(a)$ no está definida), calcula los límites laterales con `sympy` y dibuja la función.

In [ ]:
def laterales(izq_txt, der_txt, a):
    """(límite por la izquierda, límite por la derecha) de la función por partes en a."""
    fi, fd = parsear(izq_txt), parsear(der_txt)
    a = numero(a)
    return sp.limit(fi, x, a, "-"), sp.limit(fd, x, a, "+")


def veredicto(li, ld, valor=None):
    """Texto con la conclusión sobre el límite y la relación con f(a)."""
    if li.is_finite and ld.is_finite and sp.simplify(li - ld) == 0:
        txt = f"el límite existe y vale {a_texto(li)}."
        if valor is None:
            txt += " f(a) no está definida: hay un hueco."
        elif sp.simplify(valor - li) != 0:
            txt += f" Pero f(a) = {a_texto(valor)} es distinto del límite."
        else:
            txt += " Además coincide con f(a)."
        return txt
    if li == ld and li in (sp.oo, -sp.oo):
        return f"los dos lados tienden a {a_texto(li)}: el límite no existe como número (asíntota vertical)."
    return f"los laterales son distintos ({a_texto(li)} y {a_texto(ld)}): el límite no existe."


def calculadora_laterales(izq_txt, der_txt, a, valor_txt):
    try:
        li, ld = laterales(izq_txt, der_txt, a)
        valor = None if valor_txt.strip().lower() in ("no", "") else numero(valor_txt)
        a_num = float(numero(a))
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"x < {a}: f(x) = {izq_txt}      x > {a}: f(x) = {der_txt}      f({a}) = {valor_txt}")
    print("límite por la izquierda :", a_texto(li))
    print("límite por la derecha   :", a_texto(ld))
    print("conclusión              :", veredicto(li, ld, valor))
    fi = sp.lambdify(x, parsear(izq_txt), "numpy"); fd = sp.lambdify(x, parsear(der_txt), "numpy")
    xl = np.linspace(a_num - 3, a_num - 1e-3, 300); xr = np.linspace(a_num + 1e-3, a_num + 3, 300)
    with np.errstate(all="ignore"):
        yl = np.broadcast_to(fi(xl), xl.shape).astype(float); yr = np.broadcast_to(fd(xr), xr.shape).astype(float)
    yl = np.where(np.abs(yl) < 50, yl, np.nan); yr = np.where(np.abs(yr) < 50, yr, np.nan)
    plt.figure(figsize=(5, 3)); plt.plot(xl, yl, "b-", label="regla izquierda"); plt.plot(xr, yr, "b--", label="regla derecha")
    for li_, mk in ((li, "<"), (ld, ">")):
        if li_.is_finite:
            plt.plot([a_num], [float(li_)], "o", mfc="white", mec="b")
    if valor is not None:
        plt.plot([a_num], [float(valor)], "ro", label="f(a)")
    plt.axvline(a_num, color="gray", lw=0.6); plt.grid(True); plt.legend(fontsize=8)
    plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()


widgets.interact(calculadora_laterales,
    izq_txt=widgets.Text(value="x + 3", description="izquierda"),
    der_txt=widgets.Text(value="2 + (x - 1)**2/9", description="derecha"),
    a=widgets.Text(value="-2", description="a ="),
    valor_txt=widgets.Text(value="3", description="f(a) ="));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
def _lat(i, d, a):
    return laterales(i, d, a)

PRUEBAS_2 = [
    ("figura 2.2 en a = -2: izquierda 1, derecha 3", lambda: _lat("x + 3", "2 + (x - 1)**2/9", "-2") == (1, 3)),
    ("figura 2.2 en a = 1: el límite es 2 aunque g(1) = 4",
     lambda: "distinto" in veredicto(*_lat("2 + (x - 1)**2/9", "2 + (x - 1)**2/9", "1"), sp.Integer(4))),
    ("figura 2.2 en a = 4: izquierda 3, derecha +∞", lambda: _lat("2 + (x - 1)**2/9", "1/(x - 4)", "4") == (3, sp.oo)),
    ("abs(x)/x en 0: -1 y 1, no existe",
     lambda: _lat("abs(x)/x", "abs(x)/x", "0") == (-1, 1) and "no existe" in veredicto(-sp.Integer(1), sp.Integer(1))),
    ("cortante de la viga en x = 2: 8 y -4 kN, salto 12", lambda: (lambda l: l == (8, -4) and l[0] - l[1] == 12)(_lat("8", "-4", "2"))),
    ("1/(x - 2)**2 en 2: +∞ por ambos lados", lambda: _lat("1/(x - 2)**2", "1/(x - 2)**2", "2") == (sp.oo, sp.oo)),
    ("un número mal escrito se rechaza con mensaje", lambda: _rechaza(lambda: _lat("x", "x", "dos"))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Para $f(x)=x+1$ si $x<2$ y $f(x)=x^2-1$ si $x\ge 2$, calcula a mano los dos límites laterales en $a=2$ y $f(2)$. Escríbelos y compáralos con la calculadora (`x + 1`, `x**2 - 1`, `2`, `3`). Anota si el límite existe y si coincide con $f(2)$.

In [ ]:
mi_izq, mi_der, mi_valor = None, None, None      # por ejemplo: 3, 3, 3

li, ld = laterales("x + 1", "x**2 - 1", "2")
print("La calculadora da: izquierda", li, "| derecha", ld, "| f(2) =", 2**2 - 1)
for nombre, mio, ref in (("izquierda", mi_izq, li), ("derecha", mi_der, ld), ("f(2)", mi_valor, 3)):
    print(nombre, "->", "falta tu cálculo" if mio is None else ("coincide" if cerca(mio, float(ref)) else "NO coincide"))

## 3. Límites en el infinito

$\lim_{x\to\infty}f(x)=L$ significa que $f(x)$ se acerca a $L$ cuando $x$ crece sin tope; entonces la recta $y=L$ es una **asíntota horizontal**. En $-\infty$ se mira hacia la izquierda, y el resultado puede ser otro: cuidado con las raíces, porque $\sqrt{x^2}=|x|=-x$ cuando $x<0$.

La calculadora tabula $f$ en $x=\pm10,\ \pm100,\dots$, calcula el límite con `sympy` y dibuja la función con su asíntota.

In [ ]:
def limite_infinito(texto, direccion="+∞"):
    """Límite de f cuando x -> +∞ o x -> -∞."""
    if direccion not in ("+∞", "-∞"):
        raise ValueError("La dirección debe ser '+∞' o '-∞'.")
    f = parsear(texto)
    return sp.limit(f, x, sp.oo if direccion == "+∞" else -sp.oo)


def calculadora_infinito(texto, direccion, n_cifras):
    try:
        L = limite_infinito(texto, direccion)
        f = sp.lambdify(x, parsear(texto), "numpy")
    except ValueError as err:
        print("Revisa la entrada:", err); return
    s = 1 if direccion == "+∞" else -1
    print(f"f(x) = {texto}   cuando x -> {direccion}   (valores redondeados a {n_cifras} cifras significativas)")
    for k in range(1, 7):
        v = s * 10.0**k
        print(f"   x = {v:>10.0e}   f(x) = {cifras(evaluar(f, v), n_cifras)}")
    print("límite:", a_texto(L))
    if L.is_finite:
        print(f"asíntota horizontal: y = {a_texto(L)} (la curva puede cruzarla en algún x finito).")
    else:
        print("no hay asíntota horizontal de ese lado.")
    xs = np.linspace(-50, 50, 2001)
    with np.errstate(all="ignore"):
        ys = np.broadcast_to(f(xs), xs.shape).astype(float)
    ys = np.where(np.abs(ys) < 1e3, ys, np.nan)
    plt.figure(figsize=(5, 3)); plt.plot(xs, ys, "b-")
    if L.is_finite:
        plt.axhline(float(L), color="gray", ls="--", label=f"y = {a_texto(L)}"); plt.legend(fontsize=8)
    plt.grid(True); plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()


widgets.interact(calculadora_infinito,
    texto=widgets.Text(value="(3*x**2 + 2*x)/(x**2 + 4)", description="f(x) ="),
    direccion=widgets.Dropdown(options=["+∞", "-∞"], value="+∞", description="x ->"),
    n_cifras=widgets.IntSlider(value=6, min=1, max=10, description="cifras sig."));

**Valor final de una carga.** Un capacitor que se carga a través de una resistencia sigue $v(t)=V_f\,(1-e^{-t/\tau})$. Mueve $\tau$ y $V_f$ y mira cuánto falta para el valor final en $t=\tau,\ 3\tau,\ 5\tau$: la fracción no depende de $\tau$.

In [ ]:
def carga(Vf, tau):
    if tau <= 0:
        print("Revisa la entrada: la constante de tiempo debe ser positiva."); return
    for k in (1, 3, 5):
        v = Vf * (1 - math.exp(-k))
        print(f"t = {k}τ = {cifras(k*tau, 4)} s:  v = {cifras(v, 4)} V  ({cifras(100*v/Vf, 4)} % del valor final)")
    t = np.linspace(0, 6*tau, 300)
    plt.figure(figsize=(5, 3)); plt.plot(t, Vf*(1 - np.exp(-t/tau)), "b-"); plt.axhline(Vf, color="gray", ls="--")
    plt.grid(True); plt.xlabel("t (s)"); plt.ylabel("v (V)"); plt.show()


widgets.interact(carga, Vf=widgets.FloatText(value=12.0, description="V_f (V)"),
                 tau=widgets.FloatText(value=0.5, description="τ (s)"));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("(3x²+2x)/(x²+4) -> 3 en +∞ y en -∞",
     lambda: limite_infinito("(3*x**2 + 2*x)/(x**2 + 4)") == 3 and limite_infinito("(3*x**2 + 2*x)/(x**2 + 4)", "-∞") == 3),
    ("(5x+1)/(x²-2) -> 0", lambda: limite_infinito("(5*x + 1)/(x**2 - 2)") == 0),
    ("x³/(x²+1) -> +∞ (sin asíntota horizontal)", lambda: limite_infinito("x**3/(x**2 + 1)") == sp.oo),
    ("(2x+1)/sqrt(x²+1) -> 2 en +∞ y -2 en -∞",
     lambda: limite_infinito("(2*x + 1)/sqrt(x**2 + 1)") == 2 and limite_infinito("(2*x + 1)/sqrt(x**2 + 1)", "-∞") == -2),
    ("12(1 - e^(-x/0.5)) -> 12", lambda: limite_infinito("12*(1 - exp(-x/0.5))") == 12),
    ("sqrt(x²+3x) - x -> 3/2 (no 0)", lambda: limite_infinito("sqrt(x**2 + 3*x) - x") == sp.Rational(3, 2)),
    ("una dirección inválida se rechaza", lambda: _rechaza(lambda: limite_infinito("x", "infinito"))),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Calcula a mano $\displaystyle\lim_{x\to\infty}\frac{4x^2-1}{2x^2+x}$ dividiendo numerador y denominador entre $x^2$. Escribe tu resultado y compáralo con la calculadora. Anota qué término domina en cada parte de la fracción.

In [ ]:
mi_limite = None        # por ejemplo: 2

L = limite_infinito("(4*x**2 - 1)/(2*x**2 + x)")
print("La calculadora da:", L)
print("falta tu cálculo" if mi_limite is None else ("coincide" if cerca(mi_limite, float(L)) else "NO coincide: revisa qué potencia de x domina"))

## 4. Álgebra de límites y la forma indeterminada $0/0$

La estrategia del libro, en tres pasos:

1. **Sustituye** $x=a$. Si sale un número y la función es de las que se pueden evaluar directamente (polinomios, cocientes con denominador distinto de cero, raíces, exponenciales, logaritmos, seno y coseno dentro de su dominio), ese número es el límite.
2. Si sale $k/0$ con $k\neq0$, no hay límite finito: revisa los límites laterales, que suelen ser $\pm\infty$.
3. Si sale $0/0$, **todavía no sabes nada**. Factoriza y cancela, o multiplica por el conjugado si hay una raíz, y vuelve a sustituir en la expresión simplificada.

La calculadora sigue esos pasos y muestra cada uno. La simplificación que propone es una ayuda: compárala con la tuya.

In [ ]:
def _racionaliza(f):
    """Si el numerador es A + B con una raíz, multiplica arriba y abajo por A - B."""
    num, den = sp.fraction(sp.together(f))
    terminos = sp.Add.make_args(num)
    tiene_raiz = lambda t: any(p.exp == sp.Rational(1, 2) for p in t.atoms(sp.Pow))
    if len(terminos) == 2 and any(tiene_raiz(t) for t in terminos):
        R, C = terminos if tiene_raiz(terminos[0]) else terminos[::-1]   # R: término con raíz
        conj = R - C                                 # (R + C)(R - C) = R² - C²: la raíz desaparece
        return sp.expand(num * conj), den, conj
    return None


def estrategia(texto, a):
    """Devuelve (lista de pasos en texto, límite izquierdo, límite derecho)."""
    f = parsear(texto)
    a = numero(a)
    pasos = []
    valor = f.subs(x, a)
    num, den = sp.fraction(sp.together(f))
    n_a, d_a = sp.simplify(num.subs(x, a)), sp.simplify(den.subs(x, a))
    if valor.is_finite and d_a != 0:
        pasos.append(f"1. Sustitución directa: f({a}) = {a_texto(valor)}. Es un número: ese es el límite.")
    elif n_a != 0 and d_a == 0:
        pasos.append(f"1. Sustitución directa: {n_a}/0. No hay límite finito; mira los laterales (abajo).")
    elif n_a == 0 and d_a == 0:
        pasos.append("1. Sustitución directa: 0/0. Forma indeterminada: todavía no hay respuesta.")
        g = sp.cancel(f)
        if g != f and g.subs(x, a).is_finite:
            pasos.append(f"2. Factorizo: ({sp.factor(num)}) / ({sp.factor(den)}). Cancelo el factor que se anula "
                         f"(se vale porque x ≠ {a}): {g}.")
        else:
            r = _racionaliza(f)
            if r is not None:
                nuevo_num, den0, conj = r
                g = sp.cancel(nuevo_num / den0) / conj
                pasos.append(f"2. Multiplico arriba y abajo por el conjugado {conj}: el numerador queda {nuevo_num}; "
                             f"cancelo y obtengo {g}.")
            else:
                pasos.append("2. No encontré una simplificación algebraica directa; usa una tabla (sección 1) "
                             "o los límites trigonométricos (sección 5).")
        if g.subs(x, a).is_finite:
            pasos.append(f"3. Sustituyo en la expresión simplificada: {a_texto(g.subs(x, a))}.")
    else:
        pasos.append(f"1. f({a}) no está definida ({valor}). Revisa el dominio.")
    li, ld = sp.limit(f, x, a, "-"), sp.limit(f, x, a, "+")
    return pasos, li, ld


def calculadora_algebra(texto, a):
    try:
        pasos, li, ld = estrategia(texto, a)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"lím f(x) con x -> {a},   f(x) = {texto}")
    for p in pasos:
        print("  ", p)
    print(f"   Comprobación con sympy: izquierda {a_texto(li)}, derecha {a_texto(ld)}.")


widgets.interact(calculadora_algebra,
    texto=widgets.Text(value="(x**3 - 1)/(x - 1)", description="f(x) ="),
    a=widgets.Text(value="1", description="a ="));

**Una aplicación con $0/0$: la diferencia media logarítmica de temperatura.** En un intercambiador de calor se usa $\Delta T_{\mathrm{ml}}=\dfrac{\Delta T_1-\Delta T_2}{\ln(\Delta T_1/\Delta T_2)}$. Cuando las dos diferencias son iguales, la fórmula da $0/0$. La tabla muestra hacia dónde va con $\Delta T_1=20$ °C. El cálculo exacto llega en la Unidad 6.

In [ ]:
dTml = lambda d1, d2: (d1 - d2) / math.log(d1 / d2)
for d2 in (19.0, 19.9, 19.99, 20.01, 20.1, 21.0):
    print(f"ΔT2 = {d2:<6} °C   ΔT_ml = {dTml(20.0, d2):.5f} °C")

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
def _lim(texto, a):
    _, li, ld = estrategia(texto, a)
    return li if li == ld else None

PRUEBAS_4 = [
    ("(x³-1)/(x-1) en 1 da 3", lambda: _lim("(x**3 - 1)/(x - 1)", "1") == 3),
    ("(x²-9)/(x-3) en 3 da 6 (0/0 no significa 'no existe')", lambda: _lim("(x**2 - 9)/(x - 3)", "3") == 6),
    ("(sqrt(x+1)-1)/x en 0 da 1/2, con conjugado",
     lambda: _lim("(sqrt(x + 1) - 1)/x", "0") == sp.Rational(1, 2)
             and any("conjugado" in p for p in estrategia("(sqrt(x + 1) - 1)/x", "0")[0])),
    ("(sqrt(x+3)-2)/(x²-1) en 1 da 1/8 (problema DIF-U2-03)", lambda: _lim("(sqrt(x + 3) - 2)/(x**2 - 1)", "1") == sp.Rational(1, 8)),
    ("(x+1)/(x-3) en 3: 4/0, laterales -∞ y +∞",
     lambda: estrategia("(x + 1)/(x - 3)", "3")[1:] == (-sp.oo, sp.oo)),
    ("x²+1 en 2: sustitución directa da 5", lambda: "Sustitución directa: f(2) = 5" in estrategia("x**2 + 1", "2")[0][0]),
    ("ΔT_ml con ΔT2 = 19.99 está a menos de 0.01 °C de 20", lambda: abs(dTml(20.0, 19.99) - 20) < 0.01),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Calcula a mano $\displaystyle\lim_{x\to3}\frac{x^2-x-6}{x-3}$: sustituye, factoriza el numerador, cancela y vuelve a sustituir. Escribe tu resultado y compáralo con la calculadora. Anota qué factor cancelaste y por qué se vale cancelarlo.

In [ ]:
mi_limite = None        # por ejemplo: 5

ref = _lim("(x**2 - x - 6)/(x - 3)", "3")
print("La calculadora da:", ref)
print("falta tu cálculo" if mi_limite is None else ("coincide" if cerca(mi_limite, float(ref)) else "NO coincide: revisa la factorización"))

## 5. Teorema del sándwich y límites trigonométricos

El límite $\lim_{x\to0}\dfrac{\mathrm{sen}\,x}{x}=1$ vale **con $x$ en radianes**. Con $x$ en grados el mismo cociente tiende a $\pi/180\approx0.01745$. La primera calculadora tabula $\dfrac{\mathrm{sen}(kx)}{x}$ con la unidad del ángulo como control visible.

La segunda dibuja una función atrapada entre dos cotas, que es lo que pide el teorema del sándwich: si $g(x)\le f(x)\le h(x)$ cerca de $a$ y $g$ y $h$ tienden al mismo $L$, entonces $f$ también.

In [ ]:
def seno_kx_sobre_x(k, unidad, v):
    """sen(k·v)/v con v en la unidad indicada ('radianes' o 'grados')."""
    if unidad not in ("radianes", "grados"):
        raise ValueError("La unidad debe ser 'radianes' o 'grados'.")
    if v == 0:
        raise ValueError("En x = 0 el cociente no está definido; el límite se explora cerca de 0.")
    ang = k * v if unidad == "radianes" else math.radians(k * v)
    return math.sin(ang) / v


def calculadora_seno(k, unidad, n_cifras):
    try:
        seno_kx_sobre_x(k, unidad, 0.1)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    esperado = k if unidad == "radianes" else k * math.pi / 180
    print(f"sen({cifras(k, 4)}·x)/x con x en {unidad}   (valores redondeados a {n_cifras} cifras significativas)")
    for j in range(1, 6):
        v = 10.0**-j
        print(f"   x = ±{v:<8} {cifras(seno_kx_sobre_x(k, unidad, v), n_cifras):>14}  {cifras(seno_kx_sobre_x(k, unidad, -v), n_cifras):>14}")
    print(f"límite: {cifras(esperado, n_cifras)}" + ("" if unidad == "radianes" else "  (= k·π/180: la unidad cambia el resultado)"))


widgets.interact(calculadora_seno,
    k=widgets.FloatText(value=3.0, description="k ="),
    unidad=widgets.Dropdown(options=["radianes", "grados"], value="radianes", description="x en"),
    n_cifras=widgets.IntSlider(value=6, min=1, max=10, description="cifras sig."));

In [ ]:
_SANDWICH = {
    "x·sen(1/x) entre -|x| y |x|": (lambda v: v*np.sin(1/v), lambda v: -np.abs(v), lambda v: np.abs(v)),
    "x²·cos(1/x) entre -x² y x²": (lambda v: v**2*np.cos(1/v), lambda v: -v**2, lambda v: v**2),
    "sen(x)/x entre cos x y 1": (lambda v: np.sin(v)/v, lambda v: np.cos(v), lambda v: np.ones_like(v)),
}


def sandwich(caso, ancho):
    if caso not in _SANDWICH:
        print("Revisa la entrada: elige un caso del menú."); return
    if not 0 < ancho <= 2:
        print("Revisa la entrada: el ancho debe estar entre 0 y 2."); return
    f, g, h = _SANDWICH[caso]
    v = np.linspace(-ancho, ancho, 4001); v = v[np.abs(v) > 1e-6]
    ok = np.all(g(v) <= f(v) + 1e-12) and np.all(f(v) <= h(v) + 1e-12)
    print(caso, "| ¿la función queda entre las cotas en todo el intervalo dibujado?", "sí" if ok else "NO")
    plt.figure(figsize=(5, 3)); plt.plot(v, f(v), "b-", lw=0.8, label="f")
    plt.plot(v, g(v), "k:", label="cota inferior"); plt.plot(v, h(v), "k--", label="cota superior")
    plt.grid(True); plt.legend(fontsize=8); plt.xlabel("x"); plt.show()


widgets.interact(sandwich, caso=widgets.Dropdown(options=list(_SANDWICH), description="caso"),
                 ancho=widgets.FloatSlider(value=0.5, min=0.05, max=2, step=0.05, description="ancho"));

In [ ]:
# Casos de prueba de la sección 5 (resultado conocido)
PRUEBAS_5 = [
    ("sen x / x -> 1 (radianes)", lambda: sp.limit(sp.sin(x)/x, x, 0) == 1),
    ("sen(3x)/x -> 3; la tabla con x = 1e-5 da 3.0000", lambda: round(seno_kx_sobre_x(3, "radianes", 1e-5), 4) == 3.0),
    ("en grados, sen x / x -> π/180 = 0.017453", lambda: cerca(seno_kx_sobre_x(1, "grados", 1e-5), math.pi/180, 1e-9)),
    ("(1 - cos x)/x² -> 1/2 y (1 - cos x)/x -> 0",
     lambda: sp.limit((1 - sp.cos(x))/x**2, x, 0) == sp.Rational(1, 2) and sp.limit((1 - sp.cos(x))/x, x, 0) == 0),
    ("tan x / x -> 1", lambda: sp.limit(sp.tan(x)/x, x, 0) == 1),
    ("x² cos(1/x) -> 0 por sándwich", lambda: sp.limit(x**2*sp.cos(1/x), x, 0) == 0),
    ("sen(π/x) vale 0 en x = 1/n, pero no tiene límite en 0",
     lambda: all(abs(math.sin(math.pi*n)) < 1e-9 for n in (10, 100, 1000)) and isinstance(sp.limit(sp.sin(sp.pi/x), x, 0), sp.AccumBounds)),
    ("una unidad inválida se rechaza", lambda: _rechaza(lambda: seno_kx_sobre_x(1, "gradianes", 0.1))),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 5).** Calcula a mano $\displaystyle\lim_{x\to0}\frac{\mathrm{sen}\,3x}{x}$ escribiendo $\dfrac{\mathrm{sen}\,3x}{x}=3\cdot\dfrac{\mathrm{sen}\,3x}{3x}$. Escribe tu resultado y compáralo con la calculadora en radianes. Después cambia a grados y anota por qué el número cambia.

In [ ]:
mi_limite = None        # por ejemplo: 3

ref = sp.limit(sp.sin(3*x)/x, x, 0)
print("La calculadora da:", ref)
print("falta tu cálculo" if mi_limite is None else ("coincide" if cerca(mi_limite, float(ref)) else "NO coincide: ¿multiplicaste y dividiste por 3?"))

## 6. Continuidad, tipos de discontinuidad y método de bisección

$f$ es continua en $a$ si se cumplen tres condiciones: $f(a)$ está definida, $\lim_{x\to a}f(x)$ existe y los dos coinciden. Si falla alguna, la discontinuidad se clasifica con los límites laterales:

- **evitable**: los laterales coinciden y son finitos, pero $f(a)$ no existe o es otro número;
- **de salto**: los laterales son finitos y distintos;
- **infinita**: algún lateral es $\pm\infty$.

La primera calculadora clasifica el punto frontera de una función por partes. La segunda aplica el **método de bisección**, que se apoya en el teorema del valor intermedio y por eso exige que $f$ sea continua en todo $[a,b]$. Antes de iterar, la calculadora lo comprueba.

In [ ]:
def clasificar(izq_txt, der_txt, a, valor_txt):
    """'continua', 'evitable', 'de salto' o 'infinita', con los laterales."""
    li, ld = laterales(izq_txt, der_txt, a)
    valor = None if str(valor_txt).strip().lower() in ("no", "") else numero(valor_txt)
    if li.is_finite and ld.is_finite:
        if sp.simplify(li - ld) != 0:
            return "de salto", li, ld
        return ("continua" if valor is not None and sp.simplify(valor - li) == 0 else "evitable"), li, ld
    return "infinita", li, ld


def calculadora_continuidad(izq_txt, der_txt, a, valor_txt):
    try:
        tipo, li, ld = clasificar(izq_txt, der_txt, a, valor_txt)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"x < {a}: {izq_txt}    x > {a}: {der_txt}    f({a}) = {valor_txt}")
    print(f"izquierda {a_texto(li)} | derecha {a_texto(ld)}  ->  {tipo.upper()}")
    if tipo == "de salto":
        print("tamaño del salto (derecha - izquierda):", a_texto(ld - li))


widgets.interact(calculadora_continuidad,
    izq_txt=widgets.Text(value="0", description="izquierda"),
    der_txt=widgets.Text(value="10 + 18*x", description="derecha"),
    a=widgets.Text(value="0", description="a ="),
    valor_txt=widgets.Text(value="0", description="f(a) ="));

In [ ]:
def biseccion(texto, a, b, tol=1e-6, max_pasos=100):
    """(raíz aproximada, pasos, filas) o ValueError si no se cumplen las hipótesis."""
    f_expr = parsear(texto)
    a, b = float(numero(a)), float(numero(b))
    if not a < b:
        raise ValueError("Necesitas a < b.")
    if not tol > 0:
        raise ValueError("La tolerancia debe ser positiva.")
    dominio = continuous_domain(f_expr, x, sp.Interval(sp.nsimplify(a), sp.nsimplify(b)))
    if dominio != sp.Interval(sp.nsimplify(a), sp.nsimplify(b)):
        raise ValueError(f"f no es continua en todo [{cifras(a)}, {cifras(b)}] (es continua en {conjunto_a_texto(dominio)}). "
                         "El teorema del valor intermedio no aplica: un cambio de signo puede venir de un polo.")
    f = sp.lambdify(x, f_expr, "math")
    fa, fb = f(a), f(b)
    if fa * fb > 0:
        raise ValueError(f"f(a) = {cifras(fa)} y f(b) = {cifras(fb)} tienen el mismo signo: la bisección no puede empezar.")
    filas, pasos = [], 0
    while b - a > tol and pasos < max_pasos:
        c = (a + b) / 2
        filas.append((pasos, a, b, c, f(c)))
        if f(a) * f(c) <= 0: b = c
        else: a = c
        pasos += 1
    return (a + b) / 2, pasos, filas


def calculadora_biseccion(texto, a, b, tol_exp, n_cifras):
    try:
        c, n, filas = biseccion(texto, a, b, 10.0**tol_exp)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    f = sp.lambdify(x, parsear(texto), "math")
    print(f"{'paso':>4} | {'a':>12} | {'b':>12} | {'c':>12} | {'f(c)':>12}")
    for p, a_, b_, c_, fc in filas[:8]:
        print(f"{p:>4} | {cifras(a_, n_cifras):>12} | {cifras(b_, n_cifras):>12} | {cifras(c_, n_cifras):>12} | {cifras(fc, n_cifras):>12}")
    if len(filas) > 8:
        print("   ...")
    print(f"raíz ≈ {cifras(c, n_cifras)} tras {n} pasos;  f(raíz) = {f(c):.1e} (debe ser pequeño)")


widgets.interact(calculadora_biseccion,
    texto=widgets.Text(value="x**3 - x - 2", description="f(x) ="),
    a=widgets.Text(value="1", description="a ="), b=widgets.Text(value="2", description="b ="),
    tol_exp=widgets.IntSlider(value=-6, min=-12, max=-1, description="tol = 10^"),
    n_cifras=widgets.IntSlider(value=7, min=1, max=12, description="cifras sig."));

In [ ]:
# Casos de prueba de la sección 6 (resultado conocido)
PRUEBAS_6 = [
    ("x³ - x - 2 en [1, 2]: raíz 1.5214 en 20 pasos",
     lambda: (lambda r: round(r[0], 4) == 1.5214 and r[1] == 20)(biseccion("x**3 - x - 2", "1", "2"))),
    ("cos x - x en [0, 1]: raíz 0.7391 (radianes)", lambda: round(biseccion("cos(x) - x", "0", "1")[0], 4) == 0.7391),
    ("x/(x-2) - 3 en [0, 2.5] se rechaza: polo en x = 2 (Laboratorio 2.4)",
     lambda: _rechaza(lambda: biseccion("x/(x - 2) - 3", "0", "2.5"))),
    ("x/(x-2) - 3 en [2.5, 4]: raíz 3", lambda: round(biseccion("x/(x - 2) - 3", "2.5", "4")[0], 5) == 3.0),
    ("x² + 1 en [-1, 1]: sin cambio de signo, se rechaza", lambda: _rechaza(lambda: biseccion("x**2 + 1", "-1", "1"))),
    ("controlador P(e) en e = 0: salto de 10", lambda: (lambda r: r[0] == "de salto" and r[2] - r[1] == 10)(clasificar("0", "10 + 18*x", "0", "0"))),
    ("controlador P(e) en e = 5: continua", lambda: clasificar("10 + 18*x", "100", "5", "100")[0] == "continua"),
    ("(x²-1)/(x²-3x+2): evitable en 1 e infinita en 2",
     lambda: clasificar("(x**2 - 1)/(x**2 - 3*x + 2)", "(x**2 - 1)/(x**2 - 3*x + 2)", "1", "no")[0] == "evitable"
             and clasificar("(x**2 - 1)/(x**2 - 3*x + 2)", "(x**2 - 1)/(x**2 - 3*x + 2)", "2", "no")[0] == "infinita"),
]
for nombre, prueba in PRUEBAS_6:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 6).** Haz a mano dos pasos de bisección para $f(x)=x^3-x-2$ en $[1,2]$: evalúa $f$ en el punto medio, decide qué mitad conservas y repite. Escribe el intervalo que te queda y compáralo con las dos primeras filas de la calculadora.

In [ ]:
mi_intervalo = None     # por ejemplo: (1.5, 1.75)

_, _, filas = biseccion("x**3 - x - 2", "1", "2")
ref = (filas[2][1], filas[2][2])
print("La calculadora, después de dos pasos, conserva:", ref)
print("falta tu cálculo" if mi_intervalo is None else
      ("coincide" if all(cerca(u, v) for u, v in zip(mi_intervalo, ref)) else "NO coincide: revisa el signo de f en el punto medio"))

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5 + PRUEBAS_6
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")